# *Global Settings*

In [1]:
import pandas as pd
import numpy as np
import osmnx as ox
import geopandas as gpd
from pathlib import Path
import pickle
import os
import warnings
import networkx as nx

warnings.filterwarnings('ignore')

# ==============================================================================
# 0. CONFIGURACIÓN DE RUTAS PORTABLES
# ==============================================================================
BASE_DIR = Path.cwd()

# --- DEFINICIÓN DE CARPETAS --- 
INPUTS_DIR = BASE_DIR / "Inputs"
INFRA_DIR = INPUTS_DIR / "Infrastructure"
RATES_DIR = INPUTS_DIR / "Emission rates"
GPS_DIR = INPUTS_DIR / "GPS User Data"

OUTPUTS_DIR = BASE_DIR / "Outputs"
INTERMEDIATE_DIR = OUTPUTS_DIR / "Intermediate Outputs"
FINAL_DIR = OUTPUTS_DIR / "Final Outputs"

# Creation of folders
for folder in [INFRA_DIR, RATES_DIR, GPS_DIR, INTERMEDIATE_DIR, FINAL_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# --- DEFINICIÓN DE ARCHIVOS ESPECÍFICOS ---
FILE_GRAFO = INFRA_DIR / "monterrey_drive_network_V1.pkl"
FILE_GRAFO_WALK = INFRA_DIR / "monterrey_walk_network_EydanV1.pkl" 

FILE_METRO = INFRA_DIR / "lineas_metrorrey.csv"
FILE_BUS = INFRA_DIR / "rutas_buses_ZMM_oficial.geojson"
FILE_GPS_ORIGINAL = GPS_DIR / "prueba_dummy_3.parquet"
# FILE_GPS_ORIGINAL = GPS_DIR / "top_20users_1_month.parquet"
FILE_MOVES = RATES_DIR / "cleaned_emission_rates_formatted_SB.parquet"

# Archivos de Salida
FILE_INTERMEDIO = INTERMEDIATE_DIR / "Rutas_Completadas_Clasificadas.parquet"
FILE_FINAL_PARQUET = FINAL_DIR / "Resultados_Datos_Emisiones_GPS_100_all_v1.1.parquet"
FILE_FINAL_CSV = FINAL_DIR / "Resultados_Mapa_Emisiones_GPS_Kepler.csv"
FILE_FINAL_GEOJSON = FINAL_DIR / "Resultados_Mapa_Emisiones_GPS_Kepler.geojson"

# **MODULE 1: Modos de Transporte**

## M1 Input files and settings

In [2]:
import pandas as pd
import numpy as np
import geopandas as gpd
from scipy.spatial import KDTree
from shapely import wkt
import shapely.geometry

print("Cargando infraestructura de transporte...")
bus_routes = gpd.read_file(FILE_BUS)
subway_df = pd.read_csv(FILE_METRO)

# Convertimos la columna de texto a geometría espacial
if 'geometry' in subway_df.columns:
    subway_df['geometry'] = subway_df['geometry'].apply(wkt.loads)
    subway_routes = gpd.GeoDataFrame(subway_df, geometry='geometry', crs="EPSG:4326")
elif 'WKT' in subway_df.columns:
    subway_df['geometry'] = subway_df['WKT'].apply(wkt.loads)
    subway_routes = gpd.GeoDataFrame(subway_df, geometry='geometry', crs="EPSG:4326")
elif 'lat' in subway_df.columns and 'lon' in subway_df.columns:
    subway_routes = gpd.GeoDataFrame(subway_df, geometry=gpd.points_from_xy(subway_df.lon, subway_df.lat), crs="EPSG:4326")
else:
    raise ValueError("¡ALERTA! Revisa las columnas del archivo del metro.")

# ---------------------------------------------------------
# CORRECCIÓN METODOLÓGICA: PROYECCIÓN Y CACHÉ ESPACIAL
# ---------------------------------------------------------
bus_routes = bus_routes.to_crs("EPSG:32614")
subway_routes = subway_routes.to_crs("EPSG:32614")

print("Generando Índices Espaciales (R-Trees) globales...")
_ = subway_routes.sindex 
_ = bus_routes.sindex
print("Infraestructura lista.")

Cargando infraestructura de transporte...
Generando Índices Espaciales (R-Trees) globales...
Infraestructura lista.


## M1 Functions y Asignación Modal

In [3]:
import networkx as nx

def calcular_cercania_infraestructura(df, subway_routes, bus_routes):
    """Calcula cercanía al metro/bus usando distancias MÉTRICAS exactas a la línea."""
    
    # 1. Proyectar temporalmente a UTM 14N
    gdf_pts = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.longitude, df.latitude), crs="EPSG:4326")
    gdf_pts = gdf_pts.to_crs("EPSG:32614")
    
    RADIO_BUSQUEDA_METROS = 100 
    
    # 2. Búsqueda Espacial Exacta (sjoin_nearest mide a la LÍNEA, no al centroide)
    # --- METRO ---
    metro_join = gpd.sjoin_nearest(gdf_pts, subway_routes, how='left', distance_col='dist_metro')
    metro_join = metro_join[~metro_join.index.duplicated(keep='first')] # Limpiar empates de distancia
    df['near_subway_line'] = (metro_join['dist_metro'] < RADIO_BUSQUEDA_METROS).astype(int)
    
    # --- BUS ---
    bus_join = gpd.sjoin_nearest(gdf_pts, bus_routes, how='left', distance_col='dist_bus')
    bus_join = bus_join[~bus_join.index.duplicated(keep='first')] 
    df['near_bus_route'] = (bus_join['dist_bus'] < RADIO_BUSQUEDA_METROS).astype(int)
    
    return df


def clasificar_viajes(df):
    """Modelo Bayesiano completo (4 variables) original, 100% Vectorizado y Normalizado."""
    modos = ['Carro', 'Bus', 'Metro', 'Caminar']
    
    # --- MATRICES ORIGINALES (Tomadas de 4_Clasificación.ipynb) ---
    Cercania = np.array([[0.10, 0.10, 0.80, 0.00],
                        [0.10, 0.80, 0.00, 0.10],
                        [0.40, 0.25, 0.05, 0.30]])

    Velocidad = np.array([[0.05, 0.10, 0.15, 0.60],
                        [0.47, 0.38, 0.05, 0.10],
                        [0.50, 0.30, 0.20, 0.00],
                        [1.00, 0.00, 0.00, 0.00]])

    Distancia = np.array([[0.10, 0.20, 0.30, 0.40],
                        [0.25, 0.25, 0.30, 0.20],
                        [0.40, 0.15, 0.25, 0.20],
                        [0.60, 0.30, 0.00, 0.10],
                        [0.40, 0.40, 0.00, 0.20]])

    Velprom = np.array([[0.10, 0.10, 0.20, 0.60],
                        [0.40, 0.25, 0.25, 0.10]])

    # --- PREPARAR VARIABLES FALTANTES POR VIAJE ---
    # Calculamos la distancia total y velocidad promedio del viaje al vuelo para este usuario
    mask_viajes = df['trip'] > 0
    df['dist_total_km'] = 0.0
    df['avg_speed_trip'] = 0.0
    
    if mask_viajes.any():
        df.loc[mask_viajes, 'dist_total_km'] = df[mask_viajes].groupby('trip')['dis lineal [m]'].transform('sum') / 1000.0
        df.loc[mask_viajes, 'avg_speed_trip'] = df[mask_viajes].groupby('trip')['Speed [km/h]'].transform('mean')

    # --- PASO 1: ÍNDICES VECTORIZADOS (Filtros de variables) ---
    # Cercanía: 0 (Metro), 1 (Bus), 2 (Ninguno)
    idx_c = np.where(df['near_subway_line'] == 1, 0, 
            np.where(df['near_bus_route'] == 1, 1, 2))

    idx_v = np.digitize(df['Speed [km/h]'].fillna(0), bins=[6.001, 20.001, 80.001])
    idx_d = np.digitize(df['dist_total_km'], bins=[1.0, 6.001, 10.001, 18.001])
    idx_vp = np.digitize(df['avg_speed_trip'], bins=[6.001])
    
    # --- PASO 2: BAYES Y NORMALIZACIÓN POR PUNTOS ---
    # Multiplicación matricial (No normalizada)
    P_unnorm = Cercania[idx_c] * Velocidad[idx_v] * Distancia[idx_d] * Velprom[idx_vp]
    
    # Normalización: Hacemos que las probabilidades de los 4 modos sumen 1.0 en cada fila
    suma_puntos = P_unnorm.sum(axis=1, keepdims=True)
    suma_puntos[suma_puntos == 0] = 1 # Prevenir división por cero
    P_norm_puntos = P_unnorm / suma_puntos
    
    df_probs = pd.DataFrame(P_norm_puntos, columns=modos, index=df.index)
    df_probs['trip'] = df['trip']
    
    # --- PASO 3: SUMA DE PROBABILIDADES Y VOTACIÓN POR VIAJE ---
    viajes_df = df_probs[df_probs['trip'] > 0]
    
    if not viajes_df.empty:
        # Acumulamos el peso probabilístico de todos los puntos GPS dentro de un mismo viaje
        suma_por_viaje = viajes_df.groupby('trip')[modos].sum()
        # Seleccionamos el modo que obtuvo la mayor probabilidad global para ese viaje
        modos_ganadores = suma_por_viaje.idxmax(axis=1) 
        df['modo_transporte'] = df['trip'].map(modos_ganadores)
    else:
        df['modo_transporte'] = np.nan
        
    # --- PASO 4: LIMPIEZA DE PARADAS Y MEMORIA ---
    df.loc[df['trip'] <= 0, 'modo_transporte'] = 'Parada'
    df.drop(columns=['dist_total_km', 'avg_speed_trip'], inplace=True, errors='ignore')
    
    return df

# **MODULE 2: Completado de Rutas**

### M2 Input files and settings

In [4]:
import pandas as pd
import numpy as np
import osmnx as ox
import igraph as ig
import json
import geopandas as gpd
from joblib import Parallel, delayed
from datetime import timedelta
import pyproj
from shapely.ops import transform, substring
from shapely import wkt
from shapely.geometry import Point, LineString
from pyproj import Transformer
from shapely.geometry import Point, LineString
import networkx as nx
from leuvenmapmatching.matcher.distance import DistanceMatcher
from leuvenmapmatching.map.inmem import InMemMap

# ---------------------------------------------------------
# TRANSFORMADORES GLOBALES
# ---------------------------------------------------------
TRANSFORMER_TO_UTM = Transformer.from_crs("EPSG:4326", "EPSG:32614", always_xy=True)
TRANSFORMER_TO_WGS = Transformer.from_crs("EPSG:32614", "EPSG:4326", always_xy=True)

# Guardamos una versión proyectada global del metro para la interpolación
subway_routes_proj = subway_routes.to_crs("EPSG:32614")
from leuvenmapmatching.matcher.distance import DistanceMatcher
from leuvenmapmatching.map.inmem import InMemMap


### M2 Functions

In [5]:
import networkx as nx
import pandas as pd
import numpy as np
from shapely.geometry import Point, LineString
from leuvenmapmatching.matcher.distance import DistanceMatcher

print(f"Carpetas listas en: {BASE_DIR}")

# ==============================================================================
# UTILS
# ==============================================================================

def haversine_vectorized(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
    distance = 6371 * c  
    return distance

def delete_ids_with_few_rows(df, id_list, threshold=2):
    id_counts = df[df['caid'].isin(id_list)]['caid'].value_counts()
    valid_ids = id_counts[id_counts > threshold].index
    return df[df['caid'].isin(valid_ids)]

def assign_trips(df):
    """
    Segmentacion con Filtro Anti-Teletransportacion e
    Acumulador de Tiempo Estacionario (Low-Pass Filter) Corregido.
    """
    df = df.copy()
    
    if 'travel time' not in df.columns:
        if 'caid' in df.columns and 'date' in df.columns:
            df['travel time'] = df.groupby(['caid', 'date'])['local_timestamp'].shift(-1) - df['local_timestamp']
        else:
            df['travel time'] = df['local_timestamp'].shift(-1) - df['local_timestamp']

    df['Speed [km/h]'] = df['Speed [km/h]'].fillna(0.0)
    df['dis lineal [m]'] = df['dis lineal [m]'].fillna(0.0)
    df['travel time'] = df['travel time'].fillna(pd.Timedelta(seconds=0))

    speeds = df['Speed [km/h]'].tolist()
    travel_times = df['travel time'].dt.total_seconds().tolist() 
    distances = df['dis lineal [m]'].tolist()

    trips = []
    trip_counter = 0
    stop_counter = 0
    
    STOP_SPEED = 3.0  
    STOP_TIME = 300   
    T_MAX_TELEPORT = 1800 

    accumulated_stop_time = 0

    for i in range(len(speeds)):
        speed = speeds[i]
        tt = travel_times[i]
        dist = distances[i]
        
        previous_trip = trips[-1] if i > 0 else None

        if tt > T_MAX_TELEPORT:
            stop_counter -= 1
            current_trip = stop_counter
            accumulated_stop_time = 0
        else:
            if speed < STOP_SPEED and dist < 100:
                accumulated_stop_time += tt
                if accumulated_stop_time > STOP_TIME:
                    if previous_trip is None or previous_trip > 0:
                        stop_counter -= 1
                        current_trip = stop_counter
                    else:
                        current_trip = previous_trip
                else:
                    current_trip = previous_trip if previous_trip is not None else 1
            else:
                accumulated_stop_time = 0 
                if previous_trip is None or previous_trip < 0:
                    trip_counter += 1
                    current_trip = trip_counter
                else:
                    current_trip = previous_trip
        trips.append(current_trip)
    df['trip'] = trips
    return df


from leuvenmapmatching.map.inmem import InMemMap
import osmnx as ox

def nx_to_inmemmap_metric(G):
    print("Proyectando grafo a metros (UTM) para HMM...")
    G_proj = ox.project_graph(G, to_crs="EPSG:32614")
    
    # 1. INICIAR MAPA: Aquí es donde va el index_edges=True
    map_con = InMemMap(name='red_vial_mty', 
                       use_latlon=False, 
                       use_rtree=True, 
                       index_edges=True) # <-- ¡AQUÍ ESTÁ LA MAGIA!
    
    # 2. Añadir nodos: ESTRICTAMENTE (Y, X) -> (Northing, Easting)
    for node_id, data in G_proj.nodes(data=True):
        map_con.add_node(int(node_id), (float(data['y']), float(data['x'])))
        
    # 3. Añadir aristas (conservando topología compleja con keys)
    for u, v, key in G_proj.edges(keys=True):
        map_con.add_edge(int(u), int(v))
        
    # 4. Purgar la red (esto compila la red y el índice automáticamente)
    map_con.purge()
        
    return map_con, G_proj

def _generate_hmm_fallback(caid, trip, lat1, lon1, lat2, lon2, t_start, delta_t, modo):
    """Genera una fila de rescate (linea recta) si el ruteo topologico falla."""
    dist_m = haversine_vectorized(lat1, lon1, lat2, lon2) * 1000.0
    speed = (dist_m / 1000.0) / (delta_t / 3600.0) if delta_t > 0 else 0
    limite = 5.0 if modo in ['Caminar', 'Parada'] else 80.0
    flag_corr = speed > limite
    if flag_corr: 
        speed = limite
        dist_m = (speed / 3.6) * delta_t
    return {
        'caid': caid, 'trip': trip, 'latitude': lat1, 'longitude': lon1, 'Speed [km/h]': speed,
        'local_timestamp': t_start, 'start_node': 'N/A', 'end_node': 'N/A', 'osmid': 'N/A',
        'highway': 'hmm_fallback_straight', 'geometry': f'LINESTRING ({lon1} {lat1}, {lon2} {lat2})',
        'distance_m': dist_m, 'modo_transporte': modo, 'ruteo_fallido': True,
        'corregido_espacialmente': flag_corr, 'flag_auditoria': 'HMM_Fallback'
    }
    

def complete_route(id, registros_person, 
                   G_drive_proj, G_walk_proj,
                   hmm_map_drive, hmm_map_walk,
                   geometry_metro=None): # <-- Añadido para compatibilidad
    """
    Versión 3.1: Map-Matching HMM con Fallback Topológico y Reproyección Final WGS84.
    Incluye parámetro geometry_metro para evitar errores de llamada.
    """ 
    from leuvenmapmatching.matcher.distance import DistanceMatcher
    import networkx as nx
    import pandas as pd
    import geopandas as gpd
    from shapely.geometry import Point, LineString
    from pyproj import Transformer
    from shapely.ops import transform
    import osmnx as ox

    rpc_list = []
    limites_kmh = {'Caminar': 15.0,'Bus': 100.0, 'Metro': 120.0, 'Carro': 150.0}
    
    # Transformador para convertir UTM a WGS84 al final
    utm_to_wgs84 = Transformer.from_crs("EPSG:32614", "EPSG:4326", always_xy=True).transform

    for trip_id, trip_df in registros_person.groupby('trip'):
        if trip_id <= 0:
            for _, row in trip_df.iterrows():
                rpc_list.append({
                    'caid': id, 'trip': trip_id, 'latitude': row['latitude'], 'longitude': row['longitude'], 
                    'Speed [km/h]': 0.0, 'local_timestamp': row['local_timestamp'],
                    'osmid': 'N/A', 'highway': 'parada_inactiva', 
                    'geometry': f"POINT ({row['longitude']} {row['latitude']})", 
                    'distance_m': 0.0, 'modo_transporte': row['modo_transporte'], 
                    'ruteo_fallido': False, 'flag_auditoria': 'Stop'
                })
            continue

        modo_actual = trip_df['modo_transporte'].iloc[0]
        es_peaton = (str(modo_actual).lower() == 'caminar')
        G_actual = G_walk_proj if es_peaton else G_drive_proj
        map_con = hmm_map_walk if es_peaton else hmm_map_drive 
        
        # 1. PREPARACIÓN UTM
        gdf_pings_utm = gpd.GeoDataFrame(
            trip_df, 
            geometry=[Point(xy) for xy in zip(trip_df['longitude'], trip_df['latitude'])],
            crs="EPSG:4326"
        ).to_crs("EPSG:32614")
        
        obs_utm = list(zip(gdf_pings_utm.geometry.y, gdf_pings_utm.geometry.x))
        timestamps = trip_df['local_timestamp'].tolist()

        # 2. MATCHING
        matcher = DistanceMatcher(map_con, max_dist=1000, max_dist_init=1000, 
                                  min_prob_norm=0.001, obs_noise=70, latlon=False)
        
        try:
            states, _ = matcher.match(obs_utm)
            if not states or all(s is None for s in states):
                raise ValueError("HMM_FAILED")

            # 3. RECONSTRUCCIÓN CON TEJIDO TOPOLÓGICO
            for i in range(len(states) - 1):
                u_m, v_m = states[i], states[i+1]
                t_start = timestamps[i]
                delta_t = (timestamps[i+1] - t_start).total_seconds()
                
                if u_m is None or v_m is None:
                    p1_utm = Point(obs_utm[i][1], obs_utm[i][0])
                    p2_utm = Point(obs_utm[i+1][1], obs_utm[i+1][0])
                    node_u = ox.nearest_nodes(G_actual, p1_utm.x, p1_utm.y)
                    node_v = ox.nearest_nodes(G_actual, p2_utm.x, p2_utm.y)
                    try:
                        path_nodes = nx.shortest_path(G_actual, node_u, node_v, weight='length')
                        flag = 'HMM_Gap_Fixed'
                    except:
                        path_nodes = [node_u, node_v]
                        flag = 'HMM_Fallback_Straight'
                else:
                    edge_start = u_m if isinstance(u_m, tuple) else (u_m, u_m)
                    edge_end = v_m if isinstance(v_m, tuple) else (v_m, v_m)
                    try:
                        inter_path = nx.shortest_path(G_actual, edge_start[1], edge_end[0], weight='length')
                        path_nodes = [edge_start[0]] + inter_path + [edge_end[1]]
                        flag = 'HMM_Viterbi'
                    except:
                        path_nodes = [edge_start[1], edge_end[0]]
                        flag = 'HMM_Topo_Break'

                path = [path_nodes[n] for n in range(len(path_nodes)) if n == 0 or path_nodes[n] != path_nodes[n-1]]

                # 4. EXTRACCIÓN Y REPROYECCIÓN
                t_ideal_total = 0
                temp_segments = []
                for n in range(len(path)-1):
                    u, v = int(path[n]), int(path[n+1])
                    try:
                        data = min(G_actual[u][v].values(), key=lambda d: d.get('length', float('inf')))
                        t_ideal_total += data.get('travel_time', 1)
                        temp_segments.append((u, v, data))
                    except KeyError: continue 
                
                curr_t = t_start
                for u, v, data in temp_segments:
                    l_m = data.get('length', 0)
                    time_alloc = delta_t * (data.get('travel_time', 1) / t_ideal_total) if t_ideal_total > 0 else (delta_t / len(temp_segments))
                    speed_kph = (l_m / 1000.0) / (time_alloc / 3600.0) if time_alloc > 0 else 0
                    
                    max_v = limites_kmh.get(modo_actual, 150.0)
                    if speed_kph > max_v:
                        speed_kph = max_v
                        l_m = (speed_kph / 3.6) * time_alloc

                    geom_utm = data.get('geometry')
                    if geom_utm is None:
                        geom_utm = LineString([(G_actual.nodes[u]['x'], G_actual.nodes[u]['y']), 
                                               (G_actual.nodes[v]['x'], G_actual.nodes[v]['y'])])
                    
                    # Transformación a WGS84 para Kepler
                    geom_wgs84 = transform(utm_to_wgs84, geom_utm)
                    lon_final, lat_final = geom_wgs84.coords[0]

                    rpc_list.append({
                        'caid': id, 'trip': trip_id, 'latitude': lat_final, 'longitude': lon_final,
                        'Speed [km/h]': speed_kph, 'local_timestamp': curr_t, 
                        'osmid': str(data.get("osmid", "N/A")), 'highway': data.get("highway", "unclassified"),
                        'geometry': geom_wgs84.wkt, 'distance_m': l_m, 'modo_transporte': modo_actual, 
                        'ruteo_fallido': False, 'flag_auditoria': flag
                    })
                    curr_t += pd.Timedelta(seconds=time_alloc)

        except Exception as e:
            lats = trip_df['latitude'].tolist()
            lons = trip_df['longitude'].tolist()
            for i in range(len(lats) - 1):
                delta_t = (timestamps[i+1] - timestamps[i]).total_seconds()
                rpc_list.append(_generate_hmm_fallback(id, trip_id, lats[i], lons[i], lats[i+1], lons[i+1], timestamps[i], delta_t, modo_actual))

    return pd.DataFrame(rpc_list)

Carpetas listas en: c:\Users\Eydan\OneDrive\Escritorio\ITESM\MAITEC Lab\Eventos Masivos\GPS_Emissions_Project_Pipeline-v2.0


## 2. Pipeline de Ejecución

#### 2.1. GPS Data

In [ ]:
# ==============================================================================
# 1. INGESTA Y SEGMENTACIÓN TEMPORAL
# ==============================================================================
print("Cargando datos GPS de usuarios...")
df = pd.read_parquet(FILE_GPS_ORIGINAL)

# --- ZONA HORARIA (UTC -> MONTERREY) ---
print("Ajustando reloj satelital a hora local de Monterrey...")
col_tiempo = 'utc_timestamp' if 'utc_timestamp' in df.columns else 'date'

try:
    df[col_tiempo] = pd.to_datetime(df[col_tiempo], unit='s')
except ValueError:
    df[col_tiempo] = pd.to_datetime(df[col_tiempo])

if df[col_tiempo].dt.tz is None:
    df[col_tiempo] = df[col_tiempo].dt.tz_localize('UTC').dt.tz_convert('America/Monterrey')
else:
    df[col_tiempo] = df[col_tiempo].dt.tz_convert('America/Monterrey')

df[col_tiempo] = df[col_tiempo].dt.tz_localize(None)
df = df.rename(columns={col_tiempo: 'local_timestamp'})
print(f"Reloj ajustado. Columna de tiempo actualizada a: 'local_timestamp'")

# --- FILTRO DE USUARIOS ---
num_users = 1 # Cambia esto para procesar más usuarios
usuarios_unicos = df['caid'].unique()[:num_users]
df = df[df['caid'].isin(usuarios_unicos)].copy()

pings_originales = len(df)
# --- PURIFICACIÓN DE LA SEÑAL (DOWNSAMPLING ESTABILIZADO) ---
print("Estandarizando la señal temporal (10s)...")
df = df.sort_values(by=['caid', 'local_timestamp'])

df['time_bucket'] = df['local_timestamp'].dt.floor('10s')

# CORRECCIÓN: Usamos 'last' para TODO. Así le entregamos al HMM un ping 100% real, no un promedio inventado.
agg_dict = {col: 'last' for col in df.columns if col not in ['caid', 'time_bucket']}

df = df.groupby(['caid', 'time_bucket'], as_index=False).agg(agg_dict)
df = df.drop(columns=['time_bucket'])

df = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.longitude, df.latitude), crs="EPSG:4326")

# --- DELIMITACIÓN ESPACIAL (GEOFENCING) ---
ref_lat, ref_lon = 25.6866, -100.3161
threshold_distance = 48 # km
df['Distance'] = haversine_vectorized(ref_lat, ref_lon, df['latitude'].values, df['longitude'].values)
df = df[df['Distance'] < threshold_distance]

# Eliminar IDs con muy pocos datos
df = delete_ids_with_few_rows(df, df['caid'].unique(), threshold=2)

print(f"Total registros válidos post-filtrado: {len(df):,}")

# --- ESCUDO DE BORDES (DÍAS COMPLETOS) ---
df['date'] = df['local_timestamp'].dt.date
dias_unicos = sorted(df['date'].unique())

if len(dias_unicos) >= 3:
    print("Aplicando limpieza de bordes para conservar solo días 100% completos...")
    fecha_min, fecha_max = dias_unicos[0], dias_unicos[-1]
    df = df[(df['date'] > fecha_min) & (df['date'] < fecha_max)]
    print(f"Rango útil: {df['date'].min()} al {df['date'].max()}")
else:
    print("Protección de bordes omitida (pocos días en la muestra).")

print("Calculando vectores de desplazamiento y limpiando ruido...")
df = df.sort_values(by=['caid', 'local_timestamp'])

# 1. Mirada al Pasado (Llegada)
df['lat_prev'] = df.groupby(['caid', 'date'])['latitude'].shift(1)
df['lon_prev'] = df.groupby(['caid', 'date'])['longitude'].shift(1)
df['time_prev'] = df.groupby(['caid', 'date'])['local_timestamp'].shift(1)

# 2. Mirada al Futuro (Salida)
df['lat_next'] = df.groupby(['caid', 'date'])['latitude'].shift(-1)
df['lon_next'] = df.groupby(['caid', 'date'])['longitude'].shift(-1)
df['time_next'] = df.groupby(['caid', 'date'])['local_timestamp'].shift(-1)

# 3. Funciones de cálculo cinemático crudo
def calc_speed(lat1, lon1, t1, lat2, lon2, t2):
    dist_m = haversine_vectorized(lat1, lon1, lat2, lon2) * 1000
    time_s = (t2 - t1).dt.total_seconds()
    return np.where(time_s > 0, (dist_m / 1000.0) / (time_s / 3600.0), 0)

# Velocidad de llegada y velocidad de salida
df['vel_llegada'] = calc_speed(df['lat_prev'], df['lon_prev'], df['time_prev'], 
                            df['latitude'], df['longitude'], df['local_timestamp'])

df['vel_salida'] = calc_speed(df['latitude'], df['longitude'], df['local_timestamp'], 
                            df['lat_next'], df['lon_next'], df['time_next'])

condicion_glitch = (df['vel_llegada'] > 150) & ((df['vel_salida'] > 150) | df['vel_salida'].isna())

puntos_antes = len(df)
df = df[~condicion_glitch]

# 5. RECALCULAR FÍSICA LIMPIA (Sin los agujeros de los glitches)
df['lat_prev_clean'] = df.groupby(['caid', 'date'])['latitude'].shift(1)
df['lon_prev_clean'] = df.groupby(['caid', 'date'])['longitude'].shift(1)
df['time_prev_clean'] = df.groupby(['caid', 'date'])['local_timestamp'].shift(1)

df['dis lineal [m]'] = haversine_vectorized(
    df['lat_prev_clean'].values, df['lon_prev_clean'].values, 
    df['latitude'].values, df['longitude'].values
) * 1000

df['travel time_sec'] = (df['local_timestamp'] - df['time_prev_clean']).dt.total_seconds()

df['Speed [km/h]'] = np.where(
    df['travel time_sec'] > 0, 
    (df['dis lineal [m]'] / 1000.0) / (df['travel time_sec'] / 3600.0), 
    0
)

# Llenar nulos del primer punto
df['Speed [km/h]'] = df['Speed [km/h]'].fillna(0)
df['dis lineal [m]'] = df['dis lineal [m]'].fillna(0)

# Limpieza de columnas auxiliares
cols_to_drop = ['lat_prev', 'lon_prev', 'time_prev', 'lat_next', 'lon_next', 'time_next', 
                'vel_llegada', 'vel_salida', 'lat_prev_clean', 'lon_prev_clean', 'time_prev_clean', 'travel time_sec', 'Distance']
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns], errors='ignore')
df = df.reset_index(drop=True)

print(f"Glitches erradicados sin romper la ruta: {puntos_antes - len(df):,}")
print("Cálculo de vectores completado. :)")


Cargando datos GPS de usuarios...
Ajustando reloj satelital a hora local de Monterrey...
Reloj ajustado. Columna de tiempo actualizada a: 'local_timestamp'
Estandarizando la señal temporal (10s)...
Total registros válidos post-filtrado: 199
Protección de bordes omitida (pocos días en la muestra).
Calculando vectores de desplazamiento y limpiando ruido...
Glitches erradicados sin romper la ruta: 0
Cálculo de vectores completado. :)


In [7]:
import numpy as np
import pandas as pd

def modulo_1_5_asesino_outliers(df, limite_v_kmh=180, silent=True):
    """
    Filtro de higiene espacial. 
    Elimina "glitches" masivos del GPS antes de enviarlos al HMM.
    """
    if not silent:
        print(f"Iniciando Módulo 1.5: Asesino de Outliers Espaciales")
    
    # 1. Orden estricto cronológico por viaje
    df = df.sort_values(['caid', 'local_timestamp']).copy()
    
    keys_agg = ['caid']
    if 'trip' in df.columns:
        keys_agg.append('trip')

    # 2. Calcular variables cinemáticas CRUDAS (sin inventar datos)
    df['diff_time_h'] = df.groupby(keys_agg)['local_timestamp'].diff().dt.total_seconds() / 3600.0
    
    def haversine_vectorized(lat1, lon1, lat2, lon2):
        R = 6371.0
        lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
        dlat = lat2 - lat1
        dlon = lon2 - lon1
        a = np.sin(dlat/2)**2 + np.cos(lat1) * np.sin(lat2) * np.sin(dlon/2)**2
        return R * 2 * np.arcsin(np.sqrt(a))

    # Medimos sobre las coordenadas reales (latitude/longitude), no sobre suavizados
    df['dist_hav_km_prev'] = haversine_vectorized(df['latitude'].shift(), df['longitude'].shift(), df['latitude'], df['longitude'])
    df['dist_hav_km_prev'] = df['dist_hav_km_prev'].fillna(0)
    
    df['vel_hav_cruda'] = df['dist_hav_km_prev'] / df.diff_time_h.replace(0, np.nan)
    
    # 3. La Guillotina (Cualquier salto > 180 km/h se borra porque es físicamente imposible en MTY)
    mask_outlier = (df['vel_hav_cruda'] > limite_v_kmh)
    
    df_limpio = df[~mask_outlier].copy()
    
    # Limpieza de memoria
    df_limpio = df_limpio.drop(columns=['diff_time_h', 'dist_hav_km_prev', 'vel_hav_cruda'])
    
    if not silent:
        puntos_borrados = mask_outlier.sum()
        print(f"Limpieza terminada. Se eliminaron {puntos_borrados} puntos corruptos.")
        
    return df_limpio

### 2.2. Road Network Data

In [8]:
import os
import osmnx as ox
import pickle
import geopandas as gpd
from pathlib import Path

# ==============================================================================
# 1. CARGA ÚNICA DEL MAPA BASE (Desde el disco duro)
# ==============================================================================
if FILE_GRAFO.exists():
    print("Cargando grafo de AUTOS...")
    with open(FILE_GRAFO, 'rb') as f:
        G_drive = pickle.load(f)

    if FILE_GRAFO_WALK.exists():
        print("Cargando grafo de PEATONES...")
        with open(FILE_GRAFO_WALK, 'rb') as f:
            G_walk = pickle.load(f)
    else:
        print("IMPORTANT: Grafo de peatones no encontrado. Usando autos como temporal.")
        G_walk = G_drive
else:
    raise FileNotFoundError(f"No se encontró el archivo de grafo: {FILE_GRAFO}")

# ==============================================================================
# 2. PROYECCIÓN UTM Y CONSTRUCCIÓN DE MAPAS HMM (Un solo paso)
# ==============================================================================
print("Proyectando grafos a metros (UTM) y construyendo índices HMM...")
# nx_to_inmemmap_metric hace la proyección internamente y nos devuelve ambas cosas:
hmm_map_drive, G_drive_proj = nx_to_inmemmap_metric(G_drive)
hmm_map_walk, G_walk_proj = nx_to_inmemmap_metric(G_walk)

# ==============================================================================
# 3. EXTRACCIÓN DE GEODATAFRAMES (Para uso futuro)
# ==============================================================================
print("Generando GeoDataFrames de aristas...")
# Usamos las versiones que ya están proyectadas para no recalcular
edges_drive = ox.graph_to_gdfs(G_drive_proj, nodes=False).reset_index()
edges_walk = ox.graph_to_gdfs(G_walk_proj, nodes=False).reset_index()

print("Infraestructura vial unificada, proyectada y lista para el ruteo.")

Cargando grafo de AUTOS...
Cargando grafo de PEATONES...
Proyectando grafos a metros (UTM) y construyendo índices HMM...
Proyectando grafo a metros (UTM) para HMM...
Proyectando grafo a metros (UTM) para HMM...
Generando GeoDataFrames de aristas...
Infraestructura vial unificada, proyectada y lista para el ruteo.


### **DIAGNÓSTICO AISLADO DEL HMM**
Esta celda corre el ruteador para un solo viaje con máxima visibilidad de errores.

In [9]:
# ==============================================================================
# DIAGNÓSTICO AISLADO DEL HMM (UN SOLO VIAJE) - CORREGIDO A METROS
# ==============================================================================

import traceback
import math
import geopandas as gpd
from shapely.geometry import Point
from leuvenmapmatching.matcher.distance import DistanceMatcher

print("--- INICIANDO DIAGNÓSTICO AISLADO ---")

# 1. Seleccionar un viaje de prueba (el primero que se mueva)
if 'df' in locals() and not df.empty:
    # Aseguramos que tenga trips asignados (usando la función que definimos arriba)
    df_test = assign_trips(df.copy())
    trips_movimiento = df_test[df_test['trip'] > 0]
    
    if not trips_movimiento.empty:
        trip_test_id = trips_movimiento['trip'].iloc[0]
        trip_test_df = df_test[df_test['trip'] == trip_test_id]
        
        print(f"Viaje de prueba ID: {trip_test_id} ({len(trip_test_df)} puntos GPS)")
        
        # 2. Preparar observaciones (¡PROYECTADAS A METROS UTM!)
        gdf_pings = gpd.GeoDataFrame(
            trip_test_df, 
            geometry=[Point(xy) for xy in zip(trip_test_df['longitude'], trip_test_df['latitude'])],
            crs="EPSG:4326"
        ).to_crs("EPSG:32614") 
        
        # Extraemos coordenadas proyectadas (Y, X) en metros
        lats_m = gdf_pings.geometry.y.tolist()
        lons_m = gdf_pings.geometry.x.tolist()
        obs = list(zip(lats_m, lons_m))
        
        # 3. Verificación de Coordenadas
        print("\n[COORDENADAS GPS (Proyectadas a UTM EPSG:32614)]")
        print(f"  Primeros 2 puntos: {obs[:2]}")
        
        print("\n[COORDENADAS GRAFO (HMM MAP)]")
        # Extraemos un nodo aleatorio del mapa HMM de forma segura
        nodes_list = list(hmm_map_drive.all_nodes())
        sample_node_full = nodes_list[0]
        sample_node_id = sample_node_full[0] if isinstance(sample_node_full, tuple) else sample_node_full
        
        sample_node_coords = hmm_map_drive.node_coordinates(sample_node_id)
        print(f"  ID Nodo Grafo: {sample_node_id}")
        print(f"  Coordenadas Grafo (y, x): {sample_node_coords}")
        
        # Cálculo de distancia usando Pitágoras simple (porque YA ESTAMOS EN METROS PLANOS, no grados)
        dist_gps_grafo = math.sqrt((lats_m[0] - sample_node_coords[0])**2 + (lons_m[0] - sample_node_coords[1])**2)
        print(f"  Distancia GPS(0) a ese Nodo aleatorio: {dist_gps_grafo:,.2f} metros")
        
        # 4. Intento de Matching Realista
        print("\n--- Ejecutando matcher.match(obs) ---")
        # Usamos los parámetros óptimos para un entorno en metros planos
        matcher = DistanceMatcher(hmm_map_drive, max_dist=150, max_dist_init=300, obs_noise=50, latlon=False)
        
        try:
            states, last_probs = matcher.match(obs)
            if states is None or all(s is None for s in states):
                print("\nRESULTADO: El HMM no pudo encontrar ninguna secuencia de estados (None).")
                print("  Revisa si el radio de búsqueda (max_dist_init) es suficiente para el primer punto.")
            else:
                puntos_emparejados = sum(1 for s in states if s is not None)
                print(f"\nRESULTADO: Match parcial/total logrado. {puntos_emparejados}/{len(states)} puntos emparejados.")
                print(f"  Primeros 5 estados encontrados: {states[:5]}")
                
        except Exception as e:
            print("\nERROR CRÍTICO durante el matching:")
            traceback.print_exc()
    else:
        print("No se encontraron viajes en movimiento (trip > 0) para probar.")
else:
    print("El DataFrame 'df' no está disponible o está vacío.")

--- INICIANDO DIAGNÓSTICO AISLADO ---
Viaje de prueba ID: 1 (2 puntos GPS)

[COORDENADAS GPS (Proyectadas a UTM EPSG:32614)]
  Primeros 2 puntos: [(2849370.3173898472, 385341.2212468482), (2849229.9618935827, 385254.1445844814)]

[COORDENADAS GRAFO (HMM MAP)]
  ID Nodo Grafo: 744975828
  Coordenadas Grafo (y, x): (2792480.029559034, 403805.99034468597)
  Distancia GPS(0) a ese Nodo aleatorio: 59,811.81 metros

--- Ejecutando matcher.match(obs) ---

RESULTADO: Match parcial/total logrado. 1/1 puntos emparejados.
  Primeros 5 estados encontrados: [(1975557715, 1983489334)]


### 2.3. Completado de Rutas

In [ ]:
from joblib import Parallel, delayed
import pandas as pd
import numpy as np
import pickle
import traceback
from tqdm.auto import tqdm  # Librería para la barra de progreso

# ==============================================================================
# 1. FUNCIÓN ENVOLTORIO (PIPELINE M1 -> M2 -> M3) CON DEBUG
# ==============================================================================
def process_day_wrapper(id_usuario, fecha, df_dia):
    """
    Procesa un día de un usuario: Segmenta, Clasifica y Rutea.
    """
    if isinstance(df_dia, pd.Series):
        df_dia = df_dia.to_frame()

    df_dia = df_dia.reset_index(drop=True)

    if df_dia.empty:
        return pd.DataFrame() 

    try:
        # --- PASO 1: SEGMENTACIÓN (Módulo 1) ---
        df_dia = assign_trips(df_dia)
        
        # --- PASO 2: CLASIFICACIÓN MODAL (Módulo 2) ---
        if 'trip' in df_dia.columns:  
            df_dia = calcular_cercania_infraestructura(df_dia, subway_routes, bus_routes)
            df_dia = clasificar_viajes(df_dia)
        else:
            df_dia['modo_transporte'] = 'Carro' 
            
        # --- PASO 2.5: ASESINO DE OUTLIERS (Módulo 1.5) ---
        # Usamos la función del experimento para limpiar antes de rutear
        df_para_rutear = modulo_1_5_asesino_outliers(df_dia, silent=True)
        
        # --- PASO 3: COMPLETADO DE RUTAS (Módulo 3 - Híbrido HMM) ---
        # ACTUALIZADO: Usamos las variables _proj para el espacio métrico
        df_routed = complete_route(
            id=id_usuario, 
            registros_person=df_para_rutear, 
            G_drive_proj=G_drive_proj, 
            G_walk_proj=G_walk_proj, 
            hmm_map_drive=hmm_map_drive,
            hmm_map_walk=hmm_map_walk,     
            geometry_metro=subway_routes_proj if 'subway_routes_proj' in globals() else None
        )
        return df_routed

    except Exception as e:
        print(f"Error crítico en Usuario {id_usuario} (Fecha: {fecha}):")
        print(e)
        return pd.DataFrame()

# ==============================================================================
# 2. EJECUCIÓN PARALELA CON CONTADOR DE TAREAS
# ==============================================================================
# 0. Asegurar columna fecha
df['local_timestamp'] = pd.to_datetime(df['local_timestamp'])
df['date'] = df['local_timestamp'].dt.date

# PARCHE: Regenerar la columna 'travel time' para assign_trips
df['travel time'] = df.groupby(['caid', 'date'])['local_timestamp'].shift(-1) - df['local_timestamp']
df['travel time'] = df['travel time'].fillna(pd.Timedelta(seconds=0))

# --- PREPARACIÓN DE TAREAS ---
# Convertimos el groupby a una lista para conocer el número total de tareas
print("Preparando grupos de trabajo...")
grupos_tareas = list(df.groupby(['caid', 'date']))
total_tareas = len(grupos_tareas)

print(f"Iniciando procesamiento paralelo...")
print(f"Total de tareas (Usuario-Día) a procesar: {total_tareas}")

# Ejecución paralela con tqdm para ver el progreso real (X de Y)
resultados = Parallel(n_jobs=3, backend="threading")(
    delayed(process_day_wrapper)(
        id_usuario, fecha_dia, grupo_df
    ) for (id_usuario, fecha_dia), grupo_df in tqdm(grupos_tareas, desc="Procesando Pipeline")
)

# ==============================================================================
# 3. CONSOLIDACIÓN DE RESULTADOS
# ==============================================================================

print("\nAgregando resultados y consolidando topología...")

results_clean = [r for r in resultados if r is not None and not r.empty] 

if len(results_clean) > 0:
    df_completed_routes = pd.concat(results_clean, axis=0, ignore_index=True)
    
    if 'date' in df_completed_routes.columns:
        df_completed_routes = df_completed_routes.drop(columns=['date'])
    
    df_completed_routes['caid'] = df_completed_routes['caid'].astype(str)
    df_completed_routes['trip'] = df_completed_routes['trip'].astype(int)

    df_completed_routes = df_completed_routes.reset_index(drop=True)
    print(f"Ruteo finalizado. Se generaron {len(df_completed_routes):,} puntos de ruta.")
else:
    print("ERROR: No se generó ningún resultado.")
    df_completed_routes = pd.DataFrame()

Preparando grupos de trabajo...
Iniciando procesamiento paralelo...
Total de tareas (Usuario-Día) a procesar: 2


Procesando Pipeline:   0%|          | 0/2 [00:00<?, ?it/s]


Agregando resultados y consolidando topología...
Ruteo finalizado. Se generaron 197 puntos de ruta.


In [11]:
# import pandas as pd

# dfmuesra = pd.read_parquet(r"Outputs\Intermediate Outputs\Rutas_Completadas_Clasificadas.parquet")

In [12]:
# print(dfmuesra.head(10).to_markdown())

In [13]:
# # Filtrar solo los viajes reales (movimiento)
# viajes_reales = df_completed_routes[df_completed_routes['trip'] > 0]

# # Mostrar las primeras filas de los viajes
# display(viajes_reales[['trip', 'modo_transporte', 'Speed [km/h]', 'distance_m', 'osmid', 'highway', 'geometry', 'flag_auditoria']].head(10))

#### Module 2 Sanity Check

In [14]:
# ==============================================================================
# SANITY CHECK: MÓDULOS 1 Y 2 (MODOS Y RUTEO) 
# ==============================================================================

print("="*60)
print("REPORTE DE DIAGNÓSTICO: RUTAS Y MODOS")
print("="*60)

if df_completed_routes.empty:
    print("¡ADVERTENCIA! El DataFrame de rutas está vacío.")
    print("Esto puede deberse a que los usuarios seleccionados no tienen suficientes pings GPS o falló el ruteo.")
    print("="*60)
else:
    total_registros = len(df_completed_routes)
    print(f"▶ Total de puntos generados: {total_registros:,}")

    print("\n▶ 1A. Distribución Modal (Por cantidad de registros):")
    if 'modo_transporte' in df_completed_routes.columns:
        modos_pct_regs = df_completed_routes['modo_transporte'].value_counts(normalize=True) * 100
        print(modos_pct_regs.round(2).astype(str) + " %")
    else:
        print("Columna 'modo_transporte' no encontrada.")

    print("\n▶ 1B. Distribución Modal (Por distancia viajada):")
    if 'modo_transporte' in df_completed_routes.columns and 'distance_m' in df_completed_routes.columns:
        distancia_por_modo = df_completed_routes.groupby('modo_transporte')['distance_m'].sum()
        distancia_pct = (distancia_por_modo / distancia_por_modo.sum()) * 100
        print(distancia_pct.sort_values(ascending=False).round(2).astype(str) + " %")
    else:
        print("Datos de distancia o modo no disponibles.")

    print("\n▶ 2. Distribución de vías (Top 5 tipos):")
    if 'highway' in df_completed_routes.columns:
        vias_pct = df_completed_routes['highway'].value_counts(normalize=True) * 100
        print(vias_pct.head(5).round(2).astype(str) + " %")
    else:
        print("Columna 'highway' no encontrada.")

    # ... (Resto del Sanity Check envuelto en el else) ...
    print("="*60)

# ---------------------------------------------------------
# NUEVA SECCIÓN DE DIAGNÓSTICO DE RUTEO
# ---------------------------------------------------------

print("\n▶ 3. Tasa de Éxito en Completado de Rutas:")
if 'ruteo_fallido' in df_completed_routes.columns:
    # Aislamos solo los puntos en movimiento (ignoramos paradas)
    df_viajes = df_completed_routes[df_completed_routes['trip'] > 0]
    total_viajes = len(df_viajes)

    fallos_totales = df_viajes['ruteo_fallido'].sum()

    if fallos_totales > 0:
        pct_errores = (fallos_totales / total_viajes) * 100
        exito_pct = 100 - pct_errores
        print(f"  Tasa de éxito general (puros): {exito_pct:.2f}%")
        print(f"  ADVERTENCIA: {fallos_totales:,} segmentos ({pct_errores:.2f}%) fallaron el map-matching y fueron anulados.")
        
        # Desglose del tipo de error
        fallos_topologia = (df_viajes['highway'] == 'routing_error').sum()
        fallos_fisica = fallos_totales - fallos_topologia
        
        print(f"      ↳ {fallos_fisica:,} anulados por salto masivo de GPS (Teletransporte)")
        print(f"      ↳ {fallos_topologia:,} anulados por desconexión en el mapa (Topología)")
    else:
        print("  ÉXITO: 100% de los segmentos ruteados correctamente. 0 fallos.")
else:
    print("  No se encontró la columna 'ruteo_fallido' para evaluar el completado.")

# ---------------------------------------------------------
# NUEVA SECCIÓN DE AUDITORÍA DE CORRECCIÓN ESPACIAL
# ---------------------------------------------------------

print("\n▶ 4. Auditoría de Filtro Anti-Rebotes (Corrección Espacial):")
if 'corregido_espacialmente' in df_completed_routes.columns:
    corregidos = df_viajes['corregido_espacialmente'].sum()
    if total_viajes > 0:
        pct_corregidos = (corregidos / total_viajes) * 100
        print(f"   {corregidos:,} segmentos ({pct_corregidos:.2f}%) tenían exceso de velocidad y fueron rescatados.")
        print("      (Se mantuvo su dibujo en el mapa, pero se recalculó su distancia real)")
    else:
        print("  No hay viajes registrados para evaluar.")
else:
    print("  No se encontró la bandera de 'corregido_espacialmente'.")

print("="*60)

REPORTE DE DIAGNÓSTICO: RUTAS Y MODOS
▶ Total de puntos generados: 197

▶ 1A. Distribución Modal (Por cantidad de registros):
modo_transporte
Parada     91.88 %
Bus         7.11 %
Caminar     1.02 %
Name: proportion, dtype: object

▶ 1B. Distribución Modal (Por distancia viajada):
modo_transporte
Bus        98.92 %
Caminar     1.08 %
Parada       0.0 %
Name: distance_m, dtype: object

▶ 2. Distribución de vías (Top 5 tipos):
highway
parada_inactiva          91.88 %
hmm_fallback_straight     6.09 %
service                   1.02 %
residential               1.02 %
Name: proportion, dtype: object

▶ 3. Tasa de Éxito en Completado de Rutas:
  Tasa de éxito general (puros): 25.00%
  ADVERTENCIA: 12 segmentos (75.00%) fallaron el map-matching y fueron anulados.
      ↳ 12 anulados por salto masivo de GPS (Teletransporte)
      ↳ 0 anulados por desconexión en el mapa (Topología)

▶ 4. Auditoría de Filtro Anti-Rebotes (Corrección Espacial):
   0 segmentos (0.00%) tenían exceso de velocidad y f

In [15]:
import pandas as pd

# 1. Aislamos exclusivamente el % sintético
df_corregidos = df_completed_routes[df_completed_routes['corregido_espacialmente'] == True].copy()

print("============================================================")
print("AUDITORÍA DE DATOS REALES Y SINTÉTICOS")
print("============================================================")

print(f"Total de segmentos analizados: {len(df_corregidos):,}\n")

# 2. ¿De qué tamaño es la distancia que les asignamos (la sintética)?
print("▶ 1. DISTANCIA SINTÉTICA ASIGNADA (Metros):")
print(df_corregidos['distance_m'].describe(percentiles=[0.25, 0.5, 0.75, 0.90, 0.95]).round(2))

# 3. ¿A qué modos de transporte les está pasando más esto?
print("\n▶ 2. MODOS DE TRANSPORTE AFECTADOS:")
modos_afectados = df_corregidos['modo_transporte'].value_counts(normalize=True) * 100
print(modos_afectados.round(2).astype(str) + " %")

# 4. ¿Qué tipo de calles están provocando estos errores?
print("\n▶ 3. TIPO DE VIALIDAD DONDE OCURRE EL REBOTE:")
vias_afectadas = df_corregidos['highway'].value_counts(normalize=True) * 100
print(vias_afectadas.head(5).round(2).astype(str) + " %")

# ============================================================
# 5. EXPORTACIÓN PARA INSPECCIÓN VISUAL (KEPLER)
# ============================================================

ruta_muestra_visual = INTERMEDIATE_DIR / "Muestra_Tramos_Corregidos_Kepler.csv"

# Guardamos solo una muestra representativa (ej. 1000 tramos) para no saturar Kepler
df_corregidos.sample(n=min(1000, len(df_corregidos)), random_state=42).to_csv(ruta_muestra_visual, index=False)

print("\n============================================================")
print(f"ARCHIVO CREADO: {ruta_muestra_visual.name}")
print("============================================================")

AUDITORÍA DE DATOS REALES Y SINTÉTICOS
Total de segmentos analizados: 0

▶ 1. DISTANCIA SINTÉTICA ASIGNADA (Metros):
count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
90%      NaN
95%      NaN
max      NaN
Name: distance_m, dtype: float64

▶ 2. MODOS DE TRANSPORTE AFECTADOS:
Series([], Name: proportion, dtype: object)

▶ 3. TIPO DE VIALIDAD DONDE OCURRE EL REBOTE:
Series([], Name: proportion, dtype: object)

ARCHIVO CREADO: Muestra_Tramos_Corregidos_Kepler.csv


In [16]:
import pandas as pd
import numpy as np

print("============================================================")
print("AUDITORÍA DE RUTAS SINTÉTICAS (FALLBACK)")
print("============================================================")

df_auditoria = df_completed_routes.copy()

# Cambiamos a la columna correcta que detectamos en tu output
col_time = 'local_timestamp' 
col_trip = 'trip'
col_dist = 'distance_m'

# Aseguramos formato datetime
df_auditoria[col_time] = pd.to_datetime(df_auditoria[col_time])

# Calculamos la diferencia de segundos real
df_auditoria['tiempo_segundos'] = df_auditoria.groupby(['caid', col_trip])[col_time].diff().dt.total_seconds().abs().fillna(0)

df_viajes = df_auditoria[df_auditoria[col_trip] > 0].copy()
df_corregidos = df_viajes[df_viajes['corregido_espacialmente'] == True].copy()
df_puros = df_viajes[df_viajes['corregido_espacialmente'] == False].copy()

dist_pura_m = df_puros[col_dist].sum()
dist_sintetica_m = df_corregidos[col_dist].sum()
dist_total_m = dist_pura_m + dist_sintetica_m

print(f"▶ BALANCE GLOBAL DE DISTANCIAS RUTEADAS:")
print(f"  Ruteo Puro (NetworkX):   {dist_pura_m/1000:,.2f} km ({(dist_pura_m/dist_total_m)*100:.2f}%)")
print(f"  Ruteo Sintético (Topado): {dist_sintetica_m/1000:,.2f} km ({(dist_sintetica_m/dist_total_m)*100:.2f}%)\n")

df_corregidos['tipo_brecha'] = pd.cut(
    df_corregidos['tiempo_segundos'], 
    bins=[-1, 15, 60, 300, np.inf], 
    labels=['Pings Rápidos (<15s)', 'Tráfico Lento (15s-1m)', 'Hueco Moderado (1m-5m)', 'Pérdida de Señal (>5m)']
)

print("▶ 1. ANÁLISIS DE LA POBLACIÓN AFECTADA")
print("\n  ¿Cuándo falla el ruteo? (Por tamaño de brecha de tiempo):")
print((df_corregidos['tipo_brecha'].value_counts(normalize=True) * 100).round(2).astype(str) + " %")

print("\n▶ 2. DISTRIBUCIÓN DE LA DISTANCIA SINTÉTICA ASIGNADA (Metros):")
print(df_corregidos[col_dist].describe(percentiles=[0.25, 0.5, 0.75, 0.90, 0.95]).round(2))

AUDITORÍA DE RUTAS SINTÉTICAS (FALLBACK)
▶ BALANCE GLOBAL DE DISTANCIAS RUTEADAS:
  Ruteo Puro (NetworkX):   16.46 km (100.00%)
  Ruteo Sintético (Topado): 0.00 km (0.00%)

▶ 1. ANÁLISIS DE LA POBLACIÓN AFECTADA

  ¿Cuándo falla el ruteo? (Por tamaño de brecha de tiempo):
tipo_brecha
Pings Rápidos (<15s)      nan %
Tráfico Lento (15s-1m)    nan %
Hueco Moderado (1m-5m)    nan %
Pérdida de Señal (>5m)    nan %
Name: proportion, dtype: object

▶ 2. DISTRIBUCIÓN DE LA DISTANCIA SINTÉTICA ASIGNADA (Metros):
count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
90%      NaN
95%      NaN
max      NaN
Name: distance_m, dtype: float64


### 2.4. Guardado de Completado de Rutas

In [17]:
# ==============================================================================
# 5. LIMPIEZA Y GUARDADO INTELIGENTE A PARQUET
# ==============================================================================

print("Guardando rutas completadas...")

# 1. Quitar las columnas temporales del ruteo espacial
cols_a_borrar = ["nodes_drive", "nodes_walk", "date", "drive_ids", "drive_dists", "walk_ids", "walk_dists"]
df_completed_routes = df_completed_routes.drop(columns=cols_a_borrar, errors="ignore")

# 2. LA APLANADORA DE TIPOS (Directo en memoria RAM)
print("Estandarizando tipos de datos para Apache Arrow...")

# Forzar columnas de Texto (Strings puros)
cols_texto = ['caid', 'start_node', 'end_node', 'osmid', 'highway', 'geometry', 'modo_transporte']
for col in cols_texto:
    if col in df_completed_routes.columns:
        df_completed_routes[col] = df_completed_routes[col].astype(str)

# Forzar columnas Numéricas (Enteros y Floats)
if 'trip' in df_completed_routes.columns:
    df_completed_routes['trip'] = df_completed_routes['trip'].fillna(-1).astype(int)
    
cols_float = ['latitude', 'longitude', 'Speed [km/h]', 'distance_m']
for col in cols_float:
    if col in df_completed_routes.columns:
        df_completed_routes[col] = pd.to_numeric(df_completed_routes[col], errors='coerce').astype(float)

# Forzar columna de Fecha/Hora 
if 'local_timestamp' in df_completed_routes.columns:
    df_completed_routes['local_timestamp'] = pd.to_datetime(df_completed_routes['local_timestamp'], errors='coerce')
    
# Forzar Booleanos 
if 'ruteo_fallido' in df_completed_routes.columns:
    df_completed_routes['ruteo_fallido'] = df_completed_routes['ruteo_fallido'].fillna(False).astype(bool)

if 'corregido_espacialmente' in df_completed_routes.columns:
    df_completed_routes['corregido_espacialmente'] = df_completed_routes['corregido_espacialmente'].fillna(False).astype(bool)

# 3. GUARDADO FINAL (USANDO PATHLIB)
print(f"Guardando archivo en formato Parquet en la carpeta '{INTERMEDIATE_DIR.name}'...")

# Usamos la variable inteligente de tu bloque de configuración
df_completed_routes.to_parquet(FILE_INTERMEDIO, index=False)

print("Ruteo finalizado y guardado con éxito.")

Guardando rutas completadas...
Estandarizando tipos de datos para Apache Arrow...
Guardando archivo en formato Parquet en la carpeta 'Intermediate Outputs'...
Ruteo finalizado y guardado con éxito.
